# Galaxy Shapes with a Convolutional Neural Network
How elongated does a galaxy look on the sky? We'll train neural networks to measure a galaxy's **axis ratio** $q = b/a$ — round galaxies have $q \approx 1$, edge-on disks $q \approx 0.2$ — directly from its image, and compare a network that knows nothing about images with one that builds in their structure.

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

## galaxy image data

[Galaxy10 SDSS](https://astronn.readthedocs.io/en/latest/galaxy10sdss.html) (Leung & Bovy, astroNN) holds 21,785 SDSS *gri* color images, 69 × 69 pixels, each labeled by Galaxy Zoo 2 when more than 55% of volunteers agreed. We draw 2,000 galaxies from each of three groups of Galaxy Zoo classes, so the sample spans the full range of shapes. The class labels are only used here to describe the sample; the networks never see them.

| our class | Galaxy10 classes |
|---|---|
| smooth | completely round; in-between round |
| edge-on disk | edge-on with rounded bulge, boxy bulge, or no bulge |
| spiral | face-on with tight, medium, or loose spiral arms |

In [ ]:
CACHE = 'data/galaxy10_subset.npz'
GROUPS = {'smooth': [1, 2], 'edge-on disk': [4, 5, 6], 'spiral': [7, 8, 9]}

def build_cache(path=CACHE, per_class=2000, seed=2026):
    """Download Galaxy10 SDSS, merge classes, draw a balanced subsample, bin 2x2."""
    import h5py, urllib.request
    h5 = 'Galaxy10.h5'
    if not os.path.exists(h5):
        urllib.request.urlretrieve('https://www.astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5', h5)
    with h5py.File(h5, 'r') as f:
        g10 = np.array(f['ans'])
        rng = np.random.default_rng(seed)
        idx = np.concatenate([np.sort(rng.choice(np.flatnonzero(np.isin(g10, c)), per_class, replace=False))
                              for c in GROUPS.values()])
        lab = np.repeat(np.arange(len(GROUPS)), per_class)
        order = np.argsort(idx)
        idx, lab = idx[order], lab[order]
        imgs = f['images'][idx]
    binned = imgs[:, :68, :68].reshape(-1, 34, 2, 34, 2, 3).mean((2, 4)).round().astype(np.uint8)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    np.savez_compressed(path, images=binned, labels=lab.astype(np.uint8), galaxy10_class=g10[idx],
                        galaxy10_index=idx, class_names=np.array(list(GROUPS)))


if not os.path.exists(CACHE):
    build_cache()
data = np.load(CACHE)
images, labels, names = data['images'], data['labels'].astype(int), list(data['class_names'])
print(f'{len(labels)} images of {images.shape[1]} x {images.shape[2]} pixels x {images.shape[3]} bands;',
      'per class:', dict(zip(names, np.bincount(labels))))

Lets take a look at the data

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(3, 8, figsize=(11, 4.4))
for r, name in enumerate(names):
    for c, i in enumerate(rng.choice(np.flatnonzero(labels == r), 8, replace=False)):
        axes[r, c].imshow(images[i])
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(name)
plt.tight_layout()
plt.show()

## target: axis ratio
We'll try to train a neural network to predict the axis ratio, $q = b/a$, given its image. 

First, lets come up with the axis ratios. 
We can summarized the shape of the light distribution by its second moments, $Q_{xx} = \sum I\,x^2 / \sum I$, $Q_{xy}$, $Q_{yy}$ (the covariance matrix of the light). 
Then its eigenvalues are $a^2$ and $b^2$, the squared lengths of the long and short axes. 

In [ ]:
def axis_ratio(img, n_iter=20):
    ''' measure axis ratio b/a from adaptive (Gaussian-weighted) second moments of a 2-D image. 
    ''' 
    n = img.shape[0]
    y, x = np.mgrid[:n, :n] - (n - 1) / 2
    border = np.concatenate([img[0], img[-1], img[:, 0], img[:, -1]])
    I = img - np.median(border)                               # subtract the sky
    M = np.diag([9.0, 9.0])                                   # start: a round weight, sigma = 3 pixels
    for _ in range(n_iter):
        Minv = np.linalg.inv(M)
        w = I * np.exp(-0.5 * (Minv[0, 0] * x**2 + 2 * Minv[0, 1] * x * y + Minv[1, 1] * y**2))
        mx, my = (w * x).sum() / w.sum(), (w * y).sum() / w.sum()
        dx, dy = x - mx, y - my
        Q = np.array([[(w * dx * dx).sum(), (w * dx * dy).sum()],
                      [(w * dx * dy).sum(), (w * dy * dy).sum()]]) / w.sum()
        M = 2 * Q + 1e-3 * np.eye(2)                          # weight matched to the galaxy's shape
    lam = np.linalg.eigvalsh(M)
    return np.sqrt(lam[0] / lam[1])

# calculate all the axis ratios
q = np.array([axis_ratio(im.mean(axis=-1)) for im in images.astype(float)])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={'width_ratios': [1, 1.6]})
for r, name in enumerate(names):
    axes[0].hist(q[labels == r], bins=np.linspace(0, 1, 31), histtype='step', lw=1.5, label=name)
axes[0].set_xlabel('axis ratio q = b/a'); axes[0].set_ylabel('galaxies'); axes[0].legend(frameon=False, fontsize=8)
axes[1].axis('off')
order = np.argsort(q)
picks = order[np.linspace(50, len(q) - 50, 6).astype(int)]
for k, i in enumerate(picks):
    ax = fig.add_axes([0.42 + 0.095 * k, 0.25, 0.09, 0.6])
    ax.imshow(images[i]); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'q = {q[i]:.2f}', fontsize=8)
plt.show()
print(f'q ranges from {q.min():.2f} to {q.max():.2f}; mean {q.mean():.2f}, standard deviation {q.std():.2f}')

We'll use a 60/20/20 split into training, validation and test. 
Each color band is standardized/whitened with the training set's mean and standard deviation. 
PyTorch's convolution layers expect images as (batch, channel, height, width), so we move the color axis forward.

In [ ]:
X = torch.tensor(images, dtype=torch.float32).permute(0, 3, 1, 2) / 255.
Y = torch.tensor(q, dtype=torch.float32)
n = len(Y)

# shuffle the data and split 60/20/20
perm = np.random.default_rng(13).permutation(n)
i_tr, i_va, i_te = perm[:int(0.6 * n)], perm[int(0.6 * n):int(0.8 * n)], perm[int(0.8 * n):]

# whiten
band_mean = X[i_tr].mean(dim=(0, 2, 3), keepdim=True)
band_std = X[i_tr].std(dim=(0, 2, 3), keepdim=True)
X = (X - band_mean) / band_std
print(f'train {len(i_tr)}, validation {len(i_va)}, test {len(i_te)};  X has shape {tuple(X.shape)}')

## NN Training
We'll use the same training setup as in the end of the last in-class exercise with: `nn.Module`, Adam, minibatches, and early stopping on the validation loss. 
We'll use **mean squared error** loss (i.e., the negative log-likelihood of a Gaussian with fixed width). 

Since $0 < q \le 1$, we'll use a sigmoid ($\frac{1}{1+e^{-z}$) activation function in the last layer so that it only outputs a number from 0 to 1.

We'll also introduce the function `augument`, which randomly rotates the images by 0, 90, 180, 270 degrees and flips, for data augmentation (later). 

We'll assess our neural networks using two metrics: 
* `rmse`: calculates root-mean-square difference between the network's predicted axis ratio and the measured $q$
* `rotational_scatter`: calculates the scatter in predicted axis ratio when input image is rotated 0, 90, 180, 270 degrees. 

In [ ]:
def augment(x):
    ''' Apply random rotation by 0/90/180/270 degrees and random left-right flip, per minibatch.
    '''
    x = torch.rot90(x, int(np.random.randint(4)), dims=(2, 3))
    return torch.flip(x, dims=(3,)) if np.random.rand() < 0.5 else x


def rmse(model, idx):
    '''RMSE of axis ratio for galaxies in `idx`
    '''
    model.eval()
    with torch.no_grad():
        return torch.sqrt(F.mse_loss(model(X[idx]), Y[idx])).item()


def rotation_scatter(model, idx):
    ''' scatter in axis ratios when the input image is rotated in its four 90deg orientations
    '''
    model.eval()
    with torch.no_grad():
        preds = torch.stack([model(torch.rot90(X[idx], k, dims=(2, 3))) for k in range(4)])
    return preds.std(dim=0, unbiased=False).mean().item()        


def fit(model, train_idx, epochs=300, lr=1e-3, weight_decay=0.0, aug=False, patience=20, batch=64, seed=1):
    ''' function to train a given neural network `model`. We use Adam + minibatches + early stopping on the validation loss. Leaves the best weights loaded.
    '''
    torch.manual_seed(seed)
    np.random.seed(seed)

    # adam optimizer with weight decay (regularization) 
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    best_loss, best_state, best_epoch, history = np.inf, None, 0, []
    for epoch in range(epochs):
        
        model.train()
        order = torch.randperm(len(train_idx))
        train_loss = 0.0
        for k in range(0, len(train_idx), batch):
            j = train_idx[order[k:k + batch].numpy()]
            xb = augment(X[j]) if aug else X[j]
            loss = F.mse_loss(model(xb), Y[j])
            opt.zero_grad(); loss.backward(); opt.step()
            train_loss += loss.item() * len(j)
            
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(X[i_va]), Y[i_va]).item()
            
        history.append((train_loss / len(train_idx), val_loss))
        
        # early stopping 
        if val_loss < best_loss:
            best_loss, best_state, best_epoch = val_loss, copy.deepcopy(model.state_dict()), epoch
        elif epoch - best_epoch >= patience:
            break
            
    model.load_state_dict(best_state)
    return np.array(history), best_epoch

n_params = lambda model: sum(p.numel() for p in model.parameters())

## MLP on raw pixels
Flatten each image into $3 \times 34 \times 34 = 3{,}468$ numbers and feed them to a MLP with one hidden layer of 256 units and a sigmoid activation function

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, hidden=256, p_drop=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), 
                                 nn.Linear(3 * 34 * 34, hidden), 
                                 nn.ReLU(),
                                 nn.Dropout(p_drop), 
                                 nn.Linear(hidden, 1))

    def forward(self, x):
        return torch.sigmoid(self.net(x).squeeze(1))


torch.manual_seed(0)
mlp = DenseNet()

hist_mlp, ep = fit(mlp, i_tr)
print(f'MLP: {n_params(mlp):,} parameters, best epoch {ep + 1}')
print(f'   RMSE: train {rmse(mlp, i_tr):.4f}   validation {rmse(mlp, i_va):.4f}   test {rmse(mlp, i_te):.4f}')
print(f'   Rot. Scatter: train {rotation_scatter(mlp, i_tr):.4f}   valid {rotation_scatter(mlp, i_va):.4f}   test {rotation_scatter(mlp, i_te):.4f}')

check the loss curves

## visualizing a convolution
A convolution slides a small filter over the image and, at each position, takes the weighted sum of the pixels underneath. 
Let's try a fixed $3\times3$ filter that responds to horizontal changes in brightness (an edge detector) on one edge-on disk in each of two orientations.

In [ ]:
edge = torch.tensor([[-1., 0., 1.],
                     [-2., 0., 2.],
                     [-1., 0., 1.]])                    
kernel = edge.expand(1, 3, 3, 3) / 3 # reshape it for F.conv2d

# pick a edeg on disk 
i_disk = np.flatnonzero(labels == 1)[3]
x = X[i_disk:i_disk + 1]


fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for col, (img, title) in enumerate([(x, 'disk'), (torch.rot90(x, 1, (2, 3)), 'same disk, rotated 90°')]):

    # plot image
    axes[2 * col].imshow(images[i_disk] if col == 0 else np.rot90(images[i_disk], 1))

    # convolve
    im_conv = F.conv2d(img, kernel, padding=1)[0, 0]
    axes[2 * col + 1].imshow(im_conv, cmap='RdBu_r')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## convolutional neural network (CNN)
Now let's set up and train a CNN. We'll use: three blocks of (convolution, ReLU, $2\times2$ max-pooling), so that later layers see larger patches of the sky; then an average over the whole map; then a dense layer with a sigmoid

In [ ]:
class GalaxyCNN(nn.Module):
    def __init__(self, p_drop=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 34 -> 17
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 17 -> 8
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten())                         # global average pool
        self.head = nn.Sequential(nn.Dropout(p_drop), nn.Linear(64, 1))

    def forward(self, x):
        return torch.sigmoid(self.head(self.features(x)).squeeze(1))


torch.manual_seed(0)
cnn = GalaxyCNN()

hist_cnn, ep = fit(cnn, i_tr)
print(f'CNN: {n_params(cnn):,} parameters, best epoch {ep + 1} of {len(hist_cnn)}')
print(f'   RMSE: train {rmse(cnn, i_tr):.4f}   validation {rmse(cnn, i_va):.4f}   test {rmse(cnn, i_te):.4f}')
print(f'   Rot. Scatter: train {rotation_scatter(cnn, i_tr):.4f}   valid {rotation_scatter(cnn, i_va):.4f}   test {rotation_scatter(cnn, i_te):.4f}')

print(f'\ndense network: {n_params(mlp):,} parameters, test RMSE {rmse(mlp, i_te):.4f}')
print(f'CNN:           {n_params(cnn):,} parameters, test RMSE {rmse(cnn, i_te):.4f}'
      f'   ({n_params(mlp) / n_params(cnn):.0f}x fewer parameters)')

CNN has much fewer parameters but it trains much more slowly. 

How does the CNN validation loss compare to the MLP validation loss? Plot the validation loss curves

## small training set
A good architecture is like a prior that matches the problem. Like a prior, it has a larger effect when there's less data. 

Let's repeat the training, but with only 600 galaxies

In [ ]:
i_small = i_tr[:600] # use only 600 of the training galaxies

train a CNN on `i_small`. Calculate and report the RMSE and rotational scatter on the test data set 

*hint: make sure to train the CNN for enough epochs (epochs is a kwarg of the function)*

train a MLP on `i_small`. Calculate and report the RMSE and rotational scatter on the test data set. How does it compare to the CNN? 

Let's try to improve the performance of the MLP. The `fit` function has options to turn on weight decay regularization (`weight_decay`), dropout (`p_drop`), and data augmentation (`aug). Train a MLP with each of those options. Calculate and report the RMSE and rotational scatter on the test data set.

Try the same with the CNN. How does it's performance change? 